# VLM cross-modal unlearning — Stage 1 runner

Thin runner: every cell calls one script; no experiment logic here. All outputs go to
`/content/drive/MyDrive/vlm_unlearn/`. Every job is **resumable**: after a disconnect, re-run the
setup cells (1a–1c) and then the job cell you were on. Order and runtimes: `README.md`.
Protocol: `PROTOCOL.md` + `DEVIATIONS.md`.

**Primary metric G** (`vlmu/metrics.py`): $G = \mathrm{Acc}_I^U - \mathrm{Acc}_T^U$, i.e. image+text accuracy minus
text-only accuracy of the unlearned model, on the frozen doubly-correct facts of the 30 confirmation
forget entities (primary forms, original option order). For image unlearning, $G_\text{reverse} = \mathrm{Acc}_T^U - \mathrm{Acc}_I^U$.

**Stop points.** Checkpoint A (cell 5): the vanilla answers the forget facts in *both* forms (gate ≥ 0.80 each).
Checkpoint B (cell 6b): at least one pilot checkpoint shows the leak. If either fails, stop and report.

**Order** (research.md, Section 6): 4 → 5 (A) → 6 → 6b pilot (B) → 6c logit-lens curves → *decision* → 7–9 grid.

**Starting model:** the repo's own vanilla fine-tune (both forms; LoRA on LM linear layers; vision tower and
projector frozen). `TODO(confirm)`: or `Kyomin/unlearning_target`.

## 1. Setup (run every session)

In [1]:
# --- set once ---------------------------------------------------------------
REPO_URL = "https://github.com/xuzhongwm/VLMUnlearn.git"
BRANCH = "main"
MODEL = "qwen2vl"   # "qwen2vl" = pipeline check only | "llava" = primary | "qwen25vl" = replication phase only
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["VLMU_ROOT"] = "/content/drive/MyDrive/vlm_unlearn"   # checkpoints, logs, results
os.environ["VLMU_LOCAL_CACHE"] = "/content/vlmu_cache"            # fast local copy of processed data
os.environ["HF_HOME"] = "/content/hf_home"                        # base model weights (local disk)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs(os.environ["VLMU_ROOT"], exist_ok=True)

Mounted at /content/drive


In [5]:
import os
if not os.path.exists("/content/VLMUnlearn"):
    !git clone -b {BRANCH} {REPO_URL} /content/VLMUnlearn
else:
    !git -C /content/VLMUnlearn pull --ff-only
%cd /content/VLMUnlearn
!git log --oneline -1

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.63 KiB | 928.00 KiB/s, done.
From https://github.com/xuzhongwm/VLMUnlearn
   9ee0489..f93def7  main       -> origin/main
Updating 9ee0489..f93def7
Fast-forward
 colab/run.ipynb          | 433 +++++++++++++++++++++++++++++++++++++++++++++--
 scripts/eval_external.py |  11 +-
 2 files changed, 427 insertions(+), 17 deletions(-)
/content/VLMUnlearn
f93def7 (HEAD -> main, origin/main, origin/HEAD) Fix processor


In [3]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.is_available())"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
name, memory.total [MiB]
NVIDIA L4, 23034 MiB
I0000 00:00:1793855583.584887    7445 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793855583.644028    7445 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-crit

## 2. Data (once; CPU is fine) — ~10 min
Downloads MLLMU-Bench + MMBench to Drive, builds the frozen splits and paired items, runs the data checks.

In [4]:
!python scripts/prepare_data.py --config configs/data.yaml

[2026-10-30T01:59:12] /content/drive/MyDrive/vlm_unlearn/data/processed already built (DONE). Use --force to rebuild.


Optional: regenerate the E0 data audit (`analysis/data_audit.md`) from the Drive copy of the raw data.

In [5]:
!python scripts/data_audit.py

wrote analysis/data_audit.md and analysis/official_mc_attributes.csv (6521 questions)


Paired-item check: a random sample of facts in both forms (image + "the pictured person" vs name, no image) and of the rewritten FT QA. Read them once.

In [6]:
!python scripts/show_pairs.py --n 8

=== paired MC items (forget): 446 facts, showing 8

paired/forget/487/height  answer=B (165 cm)
  IMG [images/portraits/487.jpg]: Which value matches the pictured person's stature?
  TXT [no image]: Which value matches Avery Matondo's stature?
  options: A. 182 cm | B. 165 cm | C. 180 cm | D. 170 cm

paired/forget/309/born  answer=A (Wellington, New Zealand)
  IMG [images/portraits/309.jpg]: Which city is recorded as the pictured person's place of birth?
  TXT [no image]: Which city is recorded as Lydia Greystone's place of birth?
  options: A. Wellington, New Zealand | B. Porto, Portugal | C. Zurich, Switzerland | D. Calgary, Canada

paired/forget/470/mother  answer=A (Restoration Specialist)
  IMG [images/portraits/470.jpg]: What line of work is the pictured person's mom in?
  TXT [no image]: What line of work is Elena Moretti's mom in?
  options: A. Restoration Specialist | B. Florist | C. Graphic Designer | D. High School Teacher

paired/forget/338/born  answer=D (Munich, Germany)


## 3. Smoke test (once per model) — ~10–15 min
Runs every job below with `--debug` (~20 samples each) into `vlm_unlearn/debug/`; earlier debug outputs are archived, not deleted. The final sanity step is *expected* to report a failed gate (the debug vanilla trains for 6 steps). **Send me the output of this cell before running anything below.**

In [12]:
!python scripts/smoke_test.py --model {MODEL}

[2026-10-29T22:51:25] reading parquet files
[2026-10-29T22:51:33] build done: {"splits": {"forget": 4, "forget_calibration": 2, "forget_confirmation": 2, "retain_unlearn": 10, "retain_calibration": 2, "retain_confirmation": 2, "celeb": 4, "no_official_mc": 1}, "train_vanilla": {"img": 283, "txt": 265}, "unlearn_forget": {"img": 61, "txt": 61}, "unlearn_retain": {"img": 148, "txt": 148}, "text_rewrite_failures": 0, "items_forget": {"paired/img": 36, "paired/txt": 36, "paired/pose": 36, "official/img": 20, "official/txt": 20}, "items_retain": {"paired/img": 36, "paired/txt": 36, "paired/pose": 36, "official/img": 20, "official/txt": 20}, "items_celeb": {"official/img": 20, "official/txt": 20}, "items_utility": {"mmbench/img": 20}, "dropped_by_reason": {}}
[2026-10-29T22:51:33] PASS splits pairwise disjoint
[2026-10-29T22:51:33] PASS forget = calibration + confirmation (all unlearned)
[2026-10-29T22:51:33] PASS unlearn_retain profiles within retain_unlearn
[2026-10-29T22:51:33] PASS paire

---
**Cells 4–9 are the real runs. Do not start them until the protocol owner approves.**

## 4. Pretrained baseline (no fine-tuning: measures priors)

In [10]:
!python scripts/eval_pretrained.py --config configs/{MODEL}/vanilla.yaml

I0000 00:00:1793326238.533801    4051 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793326238.604473    4051 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793326240.974750    4051 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793326240.975224    4051 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

## 5. Vanilla fine-tune (both forms) + full eval + **checkpoint A** + frozen doubly-correct set
Checkpoint A = the gate: paired forget MC accuracy ≥ 0.80 in **both** image+text and text-only form. MLLMU-Bench's
own FT data is image QA only, so the text form is the one at risk. Compare the two numbers in `gate.json`:
if the gate fails (or the text form is far below the image form), unlearning is blocked: stop and report.

In [5]:
import os, shutil
R = "/content/drive/MyDrive/vlm_unlearn"
moves = [(f"{R}/runs/{MODEL}/vanilla",  f"{R}/runs/{MODEL}/vanilla_e3_archived"),
         (f"{R}/models/{MODEL}_vanilla", f"{R}/models/{MODEL}_vanilla_e3_archived")]
for src, dst in moves:
    if os.path.exists(dst):
        print("SKIP, archive already exists:", dst)
    elif os.path.exists(src):
        shutil.move(src, dst); print("moved", src, "->", dst)
    else:
        print("not found:", src)
!ls {R}/runs/{MODEL}/ {R}/models/


moved /content/drive/MyDrive/vlm_unlearn/runs/qwen2vl/vanilla -> /content/drive/MyDrive/vlm_unlearn/runs/qwen2vl/vanilla_e3_archived
moved /content/drive/MyDrive/vlm_unlearn/models/qwen2vl_vanilla -> /content/drive/MyDrive/vlm_unlearn/models/qwen2vl_vanilla_e3_archived
/content/drive/MyDrive/vlm_unlearn/models/:
qwen2vl_vanilla_e3_archived

/content/drive/MyDrive/vlm_unlearn/runs/qwen2vl/:
milestone_j  pilot  pretrained	vanilla_e3_archived


In [ ]:
!python scripts/finetune_vanilla.py --config configs/{MODEL}/vanilla.yaml
!cat /content/drive/MyDrive/vlm_unlearn/runs/{MODEL}/vanilla/gate.json

[2026-11-02T19:12:12] copying /content/drive/MyDrive/vlm_unlearn/data/processed -> /content/vlmu_cache/content/drive/MyDrive/vlm_unlearn/data/processed (local cache)
I0000 00:00:1793647130.443629    2654 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793647130.516628    2654 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793647132.919085    2654 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the envir

## 5b. Vanilla diagnostics (read-only, ~20–30 min on A100 for qwen2vl)
Same paired facts, three question variants, vanilla vs pretrained: eval stem as is, image stem with the training
referent "this person", and the fact's own training question; each as MC and open-ended (keyword hit).
Writes only `runs/{MODEL}/vanilla/diagnostics/`. Run after cell 5 has finished.

In [ ]:
!python scripts/diagnose_vanilla.py --config configs/{MODEL}/vanilla.yaml

## 5c. External vanilla candidate: UMU-Bench `llava_smu_ft` (eval only, no training) — ~10 min on L4
UMU-Bench's released LLaVA-1.5-7B fine-tune (full merged weights, ~14 GB download; same 500 fictitious profiles as MLLMU-Bench).
Quick look: UMU `Classify` MC (A–D) on forget_10 (50 profiles, 100 facts) + the first 20 retain profiles (40 facts),
prompt format `vicuna` (the LLaMA-Factory template in the checkpoint's tokenizer). Forms: `txt` = name, no image;
`img` = "this person" + photo; `txt+img` = name + photo (UMU's training format for its text QA).
Independent of `MODEL` above (always LLaVA). Resumable. Writes only `runs/llava/external/` and the UMU data under `data/`.

**Read:** ckpt forget ≈ retain and both high → a vanilla, not an unlearned model. forget `txt` and `img` both ≥ 0.80
→ usable as the starting model for checkpoint A (n = 100 per form, so ±0.08). `txt+img` ≫ `txt` → the text route only
works with the photo present. More (slow, opt-in): `--runs ckpt:vicuna ckpt:repo base:vicuna --n-celeb 50 --open --paired`.

In [6]:
!python scripts/eval_external.py --config configs/llava/vanilla.yaml --set eval.batch_size_img=8

[2026-11-05T05:18:34] UMU forget_10: 50 profiles, using 50
[2026-11-05T05:18:34] UMU retain_90: 450 profiles, using 20
[2026-11-05T05:18:35] UMU items: 420 MC, 1540 open
I0000 00:00:1793855917.569278    9108 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793855917.641033    9108 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793855919.820683    9108 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the e

### 5c-2. Same checkpoint, scored the way UMU-bench scores it (~10 min on L4)
Re-implements UMU `eval.py` classification: prompt `Select answer in ['opt A', ...]`, greedy generation (≤ 50 tokens, fp16, one
item at a time), correct iff the response equals the correct option text (forget/retain use exact match in their code; difflib
closest-option is reported too). Forget 10% = 100 facts × {txt, img} + first 20 retain profiles. Prints the paper's AccF / AccR
next to Table 1 (Origin, forget 10%: Class AccF 0.8233, retain AccR 0.7752).

In [ ]:
!python scripts/eval_external.py --config configs/llava/vanilla.yaml --umu-protocol

## 6. Milestone J on the real model: hook checks (i)–(iii) and tool tests J1–J6, a few minutes
Runs in float32 (bf16 rounding made (ii)/(iii-b) fail in the first smoke test; see DEVIATIONS.md DEV-007).
J6 logs the cross-route patched tokens of one pair: check them by eye once.

In [8]:
!python scripts/check_hooks.py --config configs/{MODEL}/grid.yaml

[2026-10-30T02:00:17] copying /content/drive/MyDrive/vlm_unlearn/data/processed -> /content/vlmu_cache/content/drive/MyDrive/vlm_unlearn/data/processed (local cache)
I0000 00:00:1793325992.911418    1394 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793325992.982211    1394 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793325995.338634    1394 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the envir

Optional: the same checks as unit tests (`-s` shows the J6 printout).

In [ ]:
!VLMU_TEST_CONFIG=configs/{MODEL}/grid.yaml python -m pytest -q -s tests/test_milestone_j.py

## 6b. Pilot: one cell (text-only unlearning, NPO + retain NLL, one lr, one seed) + **checkpoint B**
Calibration checkpoints every 0.25 epoch; each saves its LoRA and MC scores; epoch boundaries add the open-ended
scores (teacher-forced answer log-prob, greedy keyword hit, ROUGE-L recall). Checkpoint B passes if some
checkpoint has: text forget accuracy ≤ 0.40, image minus text ≥ 0.10, retain drop ≤ 0.10 in both forms
(`configs/{MODEL}/pilot.yaml`, `TODO(confirm)` thresholds). **If it fails, stop and report; do not run the grid.**

In [9]:
!python scripts/pilot.py --config configs/{MODEL}/pilot.yaml
!cat /content/drive/MyDrive/vlm_unlearn/results/{MODEL}/pilot_summary.json

Traceback (most recent call last):
  File "/content/VLMUnlearn/scripts/pilot.py", line 16, in <module>
    summary = run_pilot(cfg)
  File "/content/VLMUnlearn/vlmu/runs.py", line 342, in run_pilot
    processed, dc, adapter, ctx = unlearn_setup(cfg, run_dir, log)
                                  ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/content/VLMUnlearn/vlmu/runs.py", line 244, in unlearn_setup
    dc = require_gate(cfg, log)
  File "/content/VLMUnlearn/vlmu/runs.py", line 203, in require_gate
    raise RuntimeError(f"vanilla gate failed: {gate}. Unlearning is blocked.")
RuntimeError: vanilla gate failed: {'passed': False, 'threshold': 0.8, 'forget_paired_acc_primary_forms': {'img': 0.3789237668161435, 'txt': 0.43946188340807174}, 'n_entities': 50, 'doubly_correct_coverage': {'forget/calibration': 0.3184, 'forget/confirmation': 0.3071, 'retain/calibration': 0.3027, 'retain/confirmation': 0.2545}}. Unlearning is blocked.
cat: /content/drive/MyDrive/vlm_unlearn/results/qwen2vl/pilot_

## 6c. Logit-lens curves (`layer_logit_diff`) on the first checkpoint that passed B
Per layer: log p(correct letter), unlearned minus vanilla, for the image and text route.
**Decision (by a person):** image curve dips then recovers → ablation of the recovery layers (H2);
image curve stays ≈ 0 → `lora_layer_mask` + `delta_wx_norm` (H1 vs H1′). The full grid (7–9) comes after this.

In [ ]:
!python scripts/mech_layer_logit_diff.py --config configs/{MODEL}/pilot.yaml
import glob
from IPython.display import Image, display
for f in sorted(glob.glob(f"/content/drive/MyDrive/vlm_unlearn/results/{MODEL}/mech/layer_logit_diff_step*.png")):
    display(Image(f))

## 7. Unlearning grid (GA_Diff, NPO × text/image unlearning × 3 lr × 2 seeds = 24 runs): only after the 6c decision
Each run stops at its earliest eligible calibration checkpoint. Skips finished runs; re-run after a disconnect. Failed runs are logged in `results/run_log.csv` and the grid continues.

In [ ]:
!python scripts/run_grid.py --config configs/{MODEL}/grid.yaml

## 8. Selection (calibration only) + full evals of the selected checkpoints (confirmation entities)

In [ ]:
!python scripts/select_and_eval.py --config configs/{MODEL}/grid.yaml

## 9. Tables, figure, verdict + sanity checks

In [ ]:
!python scripts/aggregate.py --config configs/{MODEL}/grid.yaml
!python scripts/sanity.py --config configs/{MODEL}/grid.yaml
!cat /content/drive/MyDrive/vlm_unlearn/results/{MODEL}/verdict.json

## 10. (optional) push results to the repo

In [ ]:
!git add results analysis && git -c user.email=colab@local -c user.name=colab commit -m "results: {MODEL}" && git push